In [ ]:
import zipfile

with zipfile.ZipFile(
    "/content/drive/MyDrive/constitutional_relevance_xlm_roberta.zip",
    "r"
) as zip_ref:
    zip_ref.extractall(
        "constitutional_relevance_xlm_roberta"
    )

print("Extraction complete.")

Extraction complete.


In [ ]:
from transformers import pipeline
classifier = pipeline(
    "text-classification",
    model="./constitutional_relevance_xlm_roberta",
    tokenizer="constitutional_relevance_xlm_roberta",
    device=-1
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# tests= [

# ]
# for q in tests:

#     result = classifier(q)[0]

#     label = result["label"]
#     confidence = result["score"]

#     print("="*80)
#     print("Question:", q)
#     print("Prediction:", label)
#     print("Confidence:", confidence)


#     if label == "LABEL_1":

#         print("→ Constitutional question")

#         # Put your RAG pipeline here
#         # context = create_rag_context(q)
#         # answer = generate_answer(context, q)


#     else:

#         print("→ Not a constitutional question")

#         # Reject or give fallback response
#         # print("Sorry, I only answer constitutional questions.")

In [ ]:
!pip -q install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 90.5 MB/s eta 0:00:00


In [ ]:
import json
import re
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer
from google.colab import files

In [ ]:


with open("/content/drive/MyDrive/constitution_articles.json", "r", encoding="utf-8") as f:
    articles = json.load(f)

print("Total Articles:", len(articles))

In [ ]:
documents = []

for article in articles:

    document = f"""
Article Number: {article['article_number']}

Article Title: {article['article_title']}

Constitution Article:
{article['article_text']}
"""

    documents.append("passage: " + document)

print(documents[0])

In [ ]:
embedding_model = SentenceTransformer("intfloat/e5-base-v2")

In [ ]:
embeddings = embedding_model.encode(
    documents,
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print("Indexed vectors:", index.ntotal)

In [ ]:
faiss.write_index(index, "constitution.index")

with open("constitution_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        articles,
        f,
        ensure_ascii=False,
        indent=2
    )

files.download("constitution.index")
files.download("constitution_mapping.json")

In [ ]:
article_lookup = {}

for article in articles:

    article_lookup[
        article["article_number"].upper()
    ] = article

print("Lookup entries:", len(article_lookup))

In [ ]:
def find_article_number(question):

    match = re.search(
        r"\barticle\s+(\d+[A-Za-z]*)\b",
        question,
        re.IGNORECASE
    )

    if match:
        return match.group(1).upper()

    return None

In [ ]:
def create_rag_context(question, top_k=1):

    article_no = find_article_number(question)

    context = ""

    # =====================================================
    # CASE 1
    # Lookup table
    # =====================================================

    if article_no is not None and article_no in article_lookup:

        article = article_lookup[article_no]

        # print("=" * 80)
        # print("LOOKUP TABLE MATCH")
        # print("=" * 80)
        # print(f"Article Number : {article['article_number']}")
        # print(f"Article Title  : {article['article_title']}")
        # print()

        context += f"""
<ARTICLE>
Article Number: {article['article_number']}
Title: {article['article_title']}
Text:
{article['article_text']}
</ARTICLE>
"""

    # =====================================================
    # CASE 2
    # Semantic Search
    # =====================================================

    else:

        query_embedding = embedding_model.encode(
            ["query: " + question],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        D, I = index.search(
            query_embedding.astype("float32"),
            k=top_k
        )

        # print("=" * 80)
        # print(f"TOP {top_k} SEMANTIC MATCHES")
        # print("=" * 80)

        for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):

            article = articles[idx]

            # print(f"Rank       : {rank}")
            # print(f"Similarity : {score:.4f}")
            # print(f"Article    : {article['article_number']}")
            # print(f"Title      : {article['article_title']}")
            # print("-" * 80)

            context += f"""
<ARTICLE>
ARTICLE NUMBER: {article['article_number']}
Title: {article['article_title']}
Text:
   {article['article_text']}
</ARTICLE>

"""

    final_input = f"""
{context}
Question:
{question}
"""

    return final_input

In [ ]:
question = "What does state about Bangladesh ?"

context = create_rag_context(question)

# print("\n")
# print("=" * 80)
print(context)

In [ ]:
question = "Who appoints the Prime Minister article 5?"

context = create_rag_context(
    question,
    top_k=3
)

print(context)

In [ ]:
!pip -q install transformers accelerate peft sentencepiece

In [ ]:
import zipfile
import os

checkpoint_zip = "/content/drive/MyDrive/mbart_phase1_checkpoint_1152.zip"

extract_path = "./mbart_phase1_checkpoint_1152"


with zipfile.ZipFile(checkpoint_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


print("Extracted files:")
print(os.listdir(extract_path))

In [ ]:
import torch

from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast
)

from peft import (
    PeftModel,
    PeftConfig
)

In [ ]:
base_model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(
    base_model_name
)


tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "en_XX"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)


base_model = MBartForConditionalGeneration.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16
)


base_model.to(device)

In [ ]:
!pip uninstall -y torchao

In [ ]:
model = PeftModel.from_pretrained(
    base_model,
    extract_path
)


model.eval()


print("Phase 1 checkpoint loaded")

In [ ]:
def generate_answer(context):

    prompt = f"""
Context:

{context}
"""
    # print("prompt")
    # print(prompt)
    print("Model Answer")
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)


    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_length=128,
            num_beams=5,
            no_repeat_ngram_size=3,
            length_penalty=2.0,
            early_stopping=True
        )


    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

In [ ]:
context = """
<Article>
Article Number: 12
Title: Secularism and freedom of religion
Text:
        The principle of secularism shall be realised by the elimination of  (a) communalism in all its forms ;\n(b) the granting by the State of political status in favour of any religion ;\n(c) the abuse of religion for political purposes ;\n(d) any discrimination against, or persecution of, persons practicing a\nparticular religion.,
Question: What is the principle of secularism?
</Article>
"""


# question = "What is the state religion of Bangladesh?"


answer = generate_answer(
    context
)


print(answer)

In [ ]:
!pip install -q transformers==4.48.0 sentencepiece

In [ ]:
!pip uninstall -y transformers tokenizers huggingface-hub -q

In [ ]:
!pip install -q \
transformers==4.48.0 \
tokenizers==0.21.0 \
huggingface-hub==0.27.0

In [ ]:
import transformers
import tokenizers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("HF Hub:", huggingface_hub.__version__)

In [ ]:
question = "What is national bird"



In [ ]:
def constitutional_question_handler(question):

    # Run classifier
    result = classifier(question)[0]

    label = result["label"]
    confidence = result["score"]


    # ============================
    # Not Constitutional
    # ============================

    if label == "LABEL_0":

        return "Sorry, I can only answer questions related to the Constitution."



    # ============================
    # Constitutional
    # ============================

    else:

        # Retrieve context
        context = create_rag_context(question,top_k=1)


        # Generate answer using mBART
        answer = generate_answer(
            context
        )


        return  answer


In [ ]:
question = "How is prime minister appointed?"



print(constitutional_question_handler(question))

In [ ]:
!pip install -q fastapi uvicorn==0.30.6 pyngrok nest_asyncio

In [ ]:
import fastapi
import uvicorn
import pyngrok

print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)
print("pyngrok:", pyngrok.__version__)
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel


app = FastAPI(
    title="Bangladesh Constitution QA API"
)


# Allow frontend requests
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)



class Question(BaseModel):

    question: str



@app.get("/")
def home():

    return {
        "message": "Constitution QA API is running"
    }



@app.post("/ask")
def ask(data: Question):

    print("=" * 80)
    print("Incoming Question:")
    print(data.question)
    print("=" * 80)


    answer = constitutional_question_handler(
        data.question
    )


    print("Generated Answer:")
    print(answer)
    print("=" * 80)


    return {
        "question": data.question,
        "answer": answer
    }
import uvicorn
import threading


def start_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


server_thread = threading.Thread(
    target=start_server,
    daemon=True
)


server_thread.start()
from pyngrok import ngrok


ngrok.set_auth_token(
    "34nLOagzgq2VlAvLr94q9OGjmn9_3VMYmRzbnCwXcuw5JgTkZ"
)
public_url = ngrok.connect(
    8000
)


print(public_url)
import requests

response = requests.get(
    "http://127.0.0.1:8000/"
)

print(response.status_code)
print(response.json())
from pyngrok import ngrok

public_url = ngrok.connect(8000)

print(public_url)
from pyngrok import ngrok

print(ngrok.get_tunnels())
import requests

url = "https://indigenous-noncontiguously-angelo.ngrok-free.dev"

response = requests.get(url)

print(response.status_code)
print(response.text)
from pyngrok import ngrok

print(ngrok.get_tunnels())
import requests

url = "https://indigenous-noncontiguously-angelo.ngrok-free.dev"

r = requests.get(url)

print(r.status_code)
print(r.text)
import requests


url = "https://indigenous-noncontiguously-angelo.ngrok-free.dev/ask"


r = requests.post(
    url,
    json={
        "question": "What does Article 23 say?"
    }
)


print(r.status_code)
print(r.text)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')